# TP2 - Clasificación de Texto con un MLP

Dataset: **20 Newsgroups** (~18.000 posts de foros de Usenet, clasificados en 20 categorías).

Esta notebook parte de la del TP1 (Regresión Softmax): se reutiliza la carga de datos, el preprocesamiento, TF-IDF y el loop de entrenamiento.


## 0. Setup

In [ ]:
!pip install -q mlflow optuna


In [ ]:
import re
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")


## 1. Datos: 20 Newsgroups

Usamos `remove=('headers', 'footers', 'quotes')` para sacar metadata que haría el problema "demasiado fácil" (por ejemplo, el header a veces incluye el nombre del newsgroup).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))

class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"Clases ({NUM_CLASSES}):", class_names)
print(f"\nTrain: {len(newsgroups_train.data)} documentos | Test: {len(newsgroups_test.data)} documentos")

print("\n--- Ejemplo de documento ---")
print("Categoría:", class_names[newsgroups_train.target[0]])
print(newsgroups_train.data[0][:500])


In [ ]:
# Cantidad de documentos de cada clase en el set de entrenamiento
counts = np.bincount(newsgroups_train.target, minlength=NUM_CLASSES)
porcentajes = counts / counts.sum() * 100

# Cada porción se colorea según su grupo temático (el prefijo del nombre: comp, rec, sci, ...)
GROUP_COLORS = {
    "alt": "#2a78d6", "comp": "#eb6834", "misc": "#1baf7a", "rec": "#eda100",
    "sci": "#e87ba4", "soc": "#008300", "talk": "#4a3aa7",
}
colors = [GROUP_COLORS[name.split(".")[0]] for name in class_names]
labels = [f"{name}\n{c} docs ({p:.1f}%)" for name, c, p in zip(class_names, counts, porcentajes)]

fig, ax = plt.subplots(figsize=(12, 12))
ax.pie(
    counts,
    labels=labels,
    colors=colors,
    startangle=90,
    counterclock=False,
    labeldistance=1.06,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 9, "color": "#333333"},
)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), title="Grupo", loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title("Distribución de documentos por clase (train)", fontsize=14)
plt.tight_layout()
plt.show()


## 2. Preprocesamiento configurable

Vamos a armar una función de preprocesamiento con estos hiperparámetros (todos van a formar parte de la búsqueda de Optuna más adelante):

- `remove_stopwords` (`True`/`False`): sacar palabras muy frecuentes y poco informativas (`the`, `is`, `and`, ...).
- `normalization` (`"none"` / `"stem"` / `"lemmatize"`):
  - `"none"`: no se normaliza la palabra.
  - `"stem"`: **stemming** con Porter Stemmer — corta la palabra a una raíz heurística (`"running"` → `"run"`, a veces resultados no son palabras reales: `"studies"` → `"studi"`).
  - `"lemmatize"`: **lematización** con WordNet — lleva la palabra a su forma de diccionario (`"running"` → `"run"`, `"studies"` → `"study"`), más lenta pero más "correcta" lingüísticamente.

### TODO 1: completar `preprocess_text`

Pasos esperados:
1. Tokenizar el texto con `word_tokenize`.
2. Pasar todo a minúsculas y quedarse solo con tokens alfabéticos (`token.isalpha()`), para descartar números y puntuación.
3. Si `remove_stopwords` es `True`, sacar los tokens que estén en el set de stopwords.
4. Según `normalization`, aplicar `stemmer.stem(token)`, `lemmatizer.lemmatize(token)`, o dejar el token igual.
5. Devolver los tokens unidos con espacios (un string, para poder pasárselo después a `TfidfVectorizer`).

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text, remove_stopwords, normalization):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]

    # TODO: si remove_stopwords es True, filtrar los tokens que están en STOP_WORDS
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]

    # TODO: aplicar la normalización correspondiente según el valor de `normalization`
    if normalization == "stem":
        tokens = [STEMMER.stem(t) for t in tokens]
    elif normalization == "lemmatize":
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    # si normalization == "none", no se hace nada más

    return " ".join(tokens)


In [ ]:
# Sanity check: probamos las distintas combinaciones sobre una oración de ejemplo
ejemplo = "The researchers were studying the running processes and stopping conditions of the engines."
for norm in ["none", "stem", "lemmatize"]:
    for rm_sw in [False, True]:
        resultado = preprocess_text(ejemplo, remove_stopwords=rm_sw, normalization=norm)
        print(f"normalization={norm:10s} remove_stopwords={str(rm_sw):5s} -> {resultado}")


## 3. Vectorización TF-IDF

Convertimos los textos preprocesados en vectores numéricos con `TfidfVectorizer`. Como el preprocesamiento (tokenización, stopwords, normalización) ya lo hicimos nosotros, acá solo controlamos:

- `ngram_range`: `(1,1)` solo unigramas, o `(1,2)` unigramas + bigramas (captura algo de contexto, ej. `"new york"`).
- `max_features`: tamaño máximo del vocabulario (se queda con los términos más frecuentes).
- `min_df`: un término tiene que aparecer en al menos `min_df` documentos para entrar al vocabulario (filtra términos rarísimos / errores de tipeo).

### TODO 2: completar `vectorize_data`

In [ ]:
def vectorize_data(train_texts, test_texts, ngram_range, max_features, min_df):
    # TODO: crear un TfidfVectorizer con los parámetros recibidos y ajustarlo (fit) sobre train_texts,
    # después transformar train_texts y test_texts con ese mismo vectorizer.
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, min_df=min_df)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer


In [ ]:
# Sanity check rápido con pocos documentos
_sample_train = ["the cat sat on the mat", "dogs are running in the park", "cats and dogs are pets"]
_sample_test = ["the dog sat on the mat"]
_Xtr, _Xte, _vec = vectorize_data(_sample_train, _sample_test, ngram_range=(1, 1), max_features=50, min_df=2)
assert _Xtr is not None, "Completá vectorize_data"
print("Shape train:", _Xtr.shape, "| Shape test:", _Xte.shape)
print("Vocabulario (parcial):", list(_vec.vocabulary_.keys())[:10])


## 4. Modelo: MLP


In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes, activation="relu"):
        super().__init__()
        capas = []
        prev = input_dim
        for h in hidden_sizes:          # una vuelta por cada capa oculta
            capas.append(nn.Linear(prev, h))
            capas.append(ACTIVATIONS[activation]())
            prev = h
        capas.append(nn.Linear(prev, num_classes))  # salida, sin activación
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)

La clase MLP recibe el tamaño de la entrada, una lista hidden_sizes con la cantidad de neuronas de cada capa oculta, la cantidad de clases y el nombre de la función de activación. En el constructor se recorre hidden_sizes y por cada valor se agrega una capa lineal seguida de la activación elegida del diccionario ACTIVATIONS, usando prev para que cada capa conecte con el tamaño de la anterior. Al final se agrega la capa de salida, sin activación, y todas las capas se juntan con nn.Sequential, de modo que la entrada pasa por todas las capas en orden y la salida es un puntaje para cada clase.


In [ ]:
# Sanity check: una red de 2 capas ocultas
_dummy = torch.randn(4, 50)
_model_check = MLP(input_dim=50, hidden_sizes=[32, 16], num_classes=20, activation="relu")
_out = _model_check(_dummy)
assert _out.shape == (4, 20), "Revisá MLP"
print(_model_check)
print("Shape de salida OK:", _out.shape)

## 5. Entrenamiento (ya resuelto)

Dado que los datos vectorizados entran en memoria sin problema, armamos `DataLoader`s a partir de tensores densos. Logueamos loss y accuracy (train y validación) por época a TensorBoard, y a MLflow además los hiperparámetros y el modelo final.

In [ ]:
%load_ext tensorboard


In [ ]:
def to_tensor_dataset(X, y):
    X_dense = torch.tensor(X.toarray(), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_dense, y_tensor)


def train_model(model, train_loader, val_loader, optimizer, epochs, writer=None, run_name="mlp", trial=None):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                running_loss += loss.item() * xb.size(0)
                correct += (logits.argmax(dim=1) == yb).sum().item()
                total += xb.size(0)
        val_loss, val_acc = running_loss / total, correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - "
              f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        if writer is not None:
            writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
            writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

        # Pruning: si viene de Optuna, le pasamos la accuracy de validación de esta época
        # y cortamos el entrenamiento si Optuna decide que viene peor que los anteriores
        if trial is not None:
            trial.set_user_attr("history", history)
            trial.report(val_acc, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return history


## 6. Búsqueda de hiperparámetros con Optuna


In [ ]:
print("preprocesamiento para las 6 combinaciones posibles ")
preprocessed_cache = {}
for remove_stopwords in [False, True]:
    for normalization in ["none", "stem", "lemmatize"]:
        key = (remove_stopwords, normalization)
        train_proc = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_train.data]
        test_proc  = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_test.data]
        preprocessed_cache[key] = (train_proc, test_proc)
        print("  listo:", key)


In [ ]:
def objective(trial):
    # Hiperparámetros del TP1
    remove_stopwords = trial.suggest_categorical("remove_stopwords", [False, True])
    normalization = trial.suggest_categorical("normalization", ["none", "stem", "lemmatize"])
    ngram_range = trial.suggest_categorical("ngram_range", ["1gram", "1_2gram"])
    ngram_range_tuple = (1, 1) if ngram_range == "1gram" else (1, 2)
    max_features = trial.suggest_categorical("max_features", [5000, 10000, 20000])
    min_df = trial.suggest_categorical("min_df", [1, 2, 5])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # Arquitectura del MLP
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_sizes = [trial.suggest_categorical(f"n_units_l{i}", [64, 128, 256, 512]) for i in range(n_layers)]
    activation = trial.suggest_categorical("activation", ["relu", "tanh", "sigmoid"])

    # Optimizador y parámetros
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])

    train_texts, test_texts = preprocessed_cache[(remove_stopwords, normalization)]
    # se parte de una porción de los datos como validación para la búsqueda
    tr_texts, val_texts, tr_labels, val_labels = train_test_split(
        train_texts, newsgroups_train.target, test_size=0.2, random_state=SEED,
        stratify=newsgroups_train.target,
    )

    X_tr, X_val, _ = vectorize_data(tr_texts, val_texts, ngram_range_tuple, max_features, min_df)
    train_loader = DataLoader(to_tensor_dataset(X_tr, tr_labels), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(to_tensor_dataset(X_val, val_labels), batch_size=batch_size, shuffle=False)

    model = MLP(X_tr.shape[1], hidden_sizes, NUM_CLASSES, activation)

    if optimizer_name == "Adam":
        lr = trial.suggest_float("lr_Adam", 1e-4, 1e-2, log=True)
        beta1 = trial.suggest_float("adam_beta1", 0.8, 0.99)      # momento de orden 1
        beta2 = trial.suggest_float("adam_beta2", 0.9, 0.9999)    # momento de orden 2
        optimizer = optim.Adam(model.parameters(), lr=lr, betas=(beta1, beta2))
    elif optimizer_name == "RMSprop":
        lr = trial.suggest_float("lr_RMSprop", 1e-4, 1e-2, log=True)
        momentum = trial.suggest_float("rmsprop_momentum", 0.0, 0.9)  # momento de orden 1
        alpha = trial.suggest_float("rmsprop_alpha", 0.9, 0.99)       # momento de orden 2
        optimizer = optim.RMSprop(model.parameters(), lr=lr, momentum=momentum, alpha=alpha)
    else:
        lr = trial.suggest_float("lr_SGD", 1e-2, 1.0, log=True)
        momentum = trial.suggest_float("sgd_momentum", 0.0, 0.99)     # momento de orden 1
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)


    start = time.perf_counter()
    try:
        history = train_model(model, train_loader, val_loader, optimizer, epochs=5,
                              run_name=f"trial_{trial.number}", trial=trial)
    finally:
        trial.set_user_attr("train_time_s", time.perf_counter() - start)

    return history["val_acc"][-1]


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study.optimize(objective, n_trials=40)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor accuracy de validación:", study.best_value)


La función objective arma cada prueba de Optuna. Primero elige los mismos hiperparámetros del TP1 (stopwords, normalización, n-gramas, max_features, min_df y batch size), después la arquitectura del MLP (cantidad de capas ocultas, neuronas de cada capa y activación) y por último el optimizador. Cada optimizador tiene sus propios parámetros, así que Optuna elige solo los del que haya salido. Si sale Adam elige el learning rate, beta1 y beta2. Si sale RMSprop elige el learning rate, momentum y alpha. Y si sale SGD elige el learning rate y momentum. A SGD se le da un rango de learning rate más alto porque no ajusta solo el tamaño del paso. Con eso se arma el modelo, se entrena 5 épocas con el 20% del train como validación y se devuelve la accuracy de validación final, guardando además cuánto tardó. Para el pruning, la función train_model ahora recibe el trial y al terminar cada época le pasa la accuracy de validación a Optuna. El MedianPruner compara ese valor con la mediana de las pruebas anteriores en la misma época y, si viene peor, corta la prueba. Las primeras 5 pruebas y la primera época nunca se cortan, para tener con qué comparar. El sampler tiene semilla fija para que al volver a correr se prueben las mismas combinaciones.


## 7. Entrenamiento final


In [ ]:
best = study.best_params
BEST_REMOVE_SW = best["remove_stopwords"]
BEST_NORM = best["normalization"]
BEST_NGRAM = (1, 1) if best["ngram_range"] == "1gram" else (1, 2)
BEST_MAX_FEATURES = best["max_features"]
BEST_MIN_DF = best["min_df"]
BEST_BATCH_SIZE = best["batch_size"]
BEST_HIDDEN_SIZES = [best[f"n_units_l{i}"] for i in range(best["n_layers"])]
EPOCHS_FINAL = 5

train_texts_final, test_texts_final = preprocessed_cache[(BEST_REMOVE_SW, BEST_NORM)]
X_train_final, X_test_final, vectorizer_final = vectorize_data(
    train_texts_final, test_texts_final, BEST_NGRAM, BEST_MAX_FEATURES, BEST_MIN_DF
)

train_loader_final = DataLoader(to_tensor_dataset(X_train_final, newsgroups_train.target),
                                batch_size=BEST_BATCH_SIZE, shuffle=True)
test_loader_final = DataLoader(to_tensor_dataset(X_test_final, newsgroups_test.target),
                               batch_size=BEST_BATCH_SIZE, shuffle=False)

torch.manual_seed(SEED)
model = MLP(X_train_final.shape[1], BEST_HIDDEN_SIZES, NUM_CLASSES, best["activation"])

if best["optimizer"] == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=best["lr_Adam"],
                           betas=(best["adam_beta1"], best["adam_beta2"]))
elif best["optimizer"] == "RMSprop":
    optimizer = optim.RMSprop(model.parameters(), lr=best["lr_RMSprop"],
                              momentum=best["rmsprop_momentum"], alpha=best["rmsprop_alpha"])
else:
    optimizer = optim.SGD(model.parameters(), lr=best["lr_SGD"], momentum=best["sgd_momentum"])

print(model)
writer = SummaryWriter(log_dir="runs/mlp_final")

# Nos aseguramos de cerrar cualquier corrida activa antes de configurar la nueva
mlflow.end_run()
mlflow.set_experiment("mlp_20newsgroups")
with mlflow.start_run(run_name="mlp_final"):
    mlflow.log_params({**best, "epochs": EPOCHS_FINAL})
    start = time.perf_counter()
    history = train_model(model, train_loader_final, test_loader_final, optimizer,
                          epochs=EPOCHS_FINAL, writer=writer, run_name="final")
    mlp_train_time = time.perf_counter() - start
    mlflow.log_metric("train_time_s", mlp_train_time)
    # Cambiamos el formato de serialización a pickle para evitar el requerimiento de input_example
    mlflow.pytorch.log_model(model, "model", serialization_format="pickle")

writer.close()
print(f"Tiempo de entrenamiento: {mlp_train_time:.1f} s")


Se toman los mejores hiperparámetros que encontró Optuna, se vuelve a armar el TF-IDF con todo el train y se entrena el MLP con el optimizador ganador, evaluando contra el test, que no se usó en la búsqueda. Se entrena 5 épocas, las mismas que en la búsqueda, porque los hiperparámetros se eligieron para ese largo y en la búsqueda ya se vio que con más épocas el modelo empieza a memorizar el train. También se guarda cuánto tardó el entrenamiento para compararlo después con la regresión softmax.


In [ ]:
%tensorboard --logdir runs


## 8. Evaluación final


In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader_final:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Accuracy en test: {test_accuracy:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
plt.yticks(range(NUM_CLASSES), class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión - MLP sobre 20 Newsgroups")
plt.tight_layout()
plt.show()


Con el modelo ya entrenado se predice la clase de cada documento del test y se calcula la accuracy y el reporte por clase (precision, recall y f1). La matriz de confusión muestra, para cada clase real, en qué clases cayeron las predicciones, así se ve qué temas se confunden entre sí.


## 9. Análisis de todos los entrenamientos


In [ ]:
df = study.trials_dataframe()
df = df[df["state"].isin(["COMPLETE", "PRUNED"])].copy()

lr_cols = [c for c in ["params_lr_Adam", "params_lr_RMSprop", "params_lr_SGD"] if c in df]
unit_cols = [c for c in ["params_n_units_l0", "params_n_units_l1", "params_n_units_l2"] if c in df]

df["Estado"] = df["state"].map({"COMPLETE": "Completo", "PRUNED": "Podado"})
df["Accuracy de validación"] = df["value"]
df["Optimizador"] = df["params_optimizer"]
df["Activación"] = df["params_activation"]
df["Capas ocultas"] = df["params_n_layers"]
df["Neuronas"] = df[unit_cols].apply(lambda r: "-".join(str(int(v)) for v in r if pd.notna(v)), axis=1)
df["Learning rate"] = df[lr_cols].bfill(axis=1).iloc[:, 0]
df["Batch size"] = df["params_batch_size"]
df["Max features"] = df["params_max_features"]
df["Épocas"] = df["user_attrs_history"].apply(lambda h: len(h["val_acc"]))
df["Tiempo (s)"] = df["user_attrs_train_time_s"]
df["Tiempo por época (s)"] = df["Tiempo (s)"] / df["Épocas"]

completos = df[df["Estado"] == "Completo"]
podados = df[df["Estado"] == "Podado"]

print(f"Trials: {len(df)} | completos: {len(completos)} | podados: {len(podados)}")
print(f"Tiempo total entrenando: {df['Tiempo (s)'].sum() / 60:.1f} min "
      f"(completos {completos['Tiempo (s)'].sum() / 60:.1f} min, podados {podados['Tiempo (s)'].sum() / 60:.1f} min)")
print(f"Épocas ahorradas por el pruning: {(5 - podados['Épocas']).sum()} de {len(df) * 5}")

cols = ["Accuracy de validación", "Optimizador", "Learning rate", "Activación", "Capas ocultas",
        "Neuronas", "Batch size", "Max features", "Tiempo (s)"]
print("\nLos 10 mejores trials:")
display(completos.sort_values("Accuracy de validación", ascending=False)[cols].head(10).round(4))


Se pasan todos los trials de Optuna a una tabla, con una columna por hiperparámetro, si el trial se completó o se cortó por pruning, cuántas épocas llegó a entrenar y cuánto tardó. Como cada optimizador guarda su learning rate con otro nombre, se juntan en una sola columna. Después se imprime cuántos trials se cortaron, cuánto tiempo se fue en cada grupo y cuántas épocas se ahorraron, y se muestran los 10 mejores trials.


In [ ]:
# Accuracy de validación según optimizador, activación, cantidad de capas y learning rate (solo trials completos)
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, col in zip(axes.flatten()[:3], ["Optimizador", "Activación", "Capas ocultas"]):
    sns.boxplot(data=completos, x=col, y="Accuracy de validación", ax=ax, color="#cfe0f3")
    sns.stripplot(data=completos, x=col, y="Accuracy de validación", ax=ax, color="black", alpha=0.6, size=6)
    ax.set_title(f"Accuracy de validación según {col.lower()}")
    ax.grid(axis="y", linestyle="--", alpha=0.7)

ax = axes[1, 1]
sns.scatterplot(data=completos, x="Learning rate", y="Accuracy de validación", hue="Optimizador", s=70, ax=ax)
ax.set_xscale("log")
ax.set_title("Accuracy de validación según learning rate")
ax.grid(linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()


Estos gráficos comparan la accuracy de validación según el optimizador, la activación, la cantidad de capas ocultas y el learning rate. Solo se usan los trials completos, porque los podados no llegaron a las 5 épocas y su accuracy no es comparable. Cada punto negro es un trial y la caja muestra dónde está la mayoría.


In [ ]:
# Tiempos de entrenamiento
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

ax = axes[0]
sns.scatterplot(data=completos, x="Tiempo (s)", y="Accuracy de validación", hue="Optimizador", s=70, ax=ax)
ax.set_title("Accuracy vs tiempo de entrenamiento\n(trials completos)")
ax.grid(linestyle="--", alpha=0.7)

ax = axes[1]
sns.boxplot(data=df, x="Capas ocultas", y="Tiempo por época (s)", ax=ax, color="#cfe0f3")
sns.stripplot(data=df, x="Capas ocultas", y="Tiempo por época (s)", ax=ax, color="black", alpha=0.6, size=5)
ax.set_title("Tiempo por época según capas ocultas\n(todos los trials)")
ax.grid(axis="y", linestyle="--", alpha=0.7)

ax = axes[2]
sns.boxplot(data=df, x="Batch size", y="Tiempo por época (s)", ax=ax, color="#cfe0f3")
sns.stripplot(data=df, x="Batch size", y="Tiempo por época (s)", ax=ax, color="black", alpha=0.6, size=5)
ax.set_title("Tiempo por época según batch size\n(todos los trials)")
ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()


Acá se comparan los tiempos. El primer gráfico muestra si los trials que más tardaron fueron también los que mejor anduvieron. Los otros dos usan el tiempo por época, que sirve para comparar también los trials podados, y muestran cuánto influyen la cantidad de capas y el batch size en lo que tarda cada época.


In [ ]:
# Curva de accuracy de validación de cada trial, separada por optimizador. En gris los que cortó el pruning.
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, opt_name in zip(axes, ["Adam", "RMSprop", "SGD"]):
    for t in study.trials:
        if t.params.get("optimizer") != opt_name or "history" not in t.user_attrs:
            continue
        val_acc = t.user_attrs["history"]["val_acc"]
        podado = t.state == optuna.trial.TrialState.PRUNED
        ax.plot(range(1, len(val_acc) + 1), val_acc, marker="o", ms=3,
                color="#bbbbbb" if podado else "#2a78d6", alpha=0.8, zorder=1 if podado else 2)
    ax.set_title(opt_name)
    ax.set_xlabel("Época")
    ax.set_xticks(range(1, 6))
    ax.grid(linestyle="--", alpha=0.7)
axes[0].set_ylabel("Accuracy de validación")
axes[0].legend(handles=[Line2D([], [], color="#2a78d6", marker="o", label="Completo"),
                        Line2D([], [], color="#bbbbbb", marker="o", label="Podado")])
fig.suptitle("Accuracy de validación por época de todos los trials", fontsize=14)
plt.tight_layout()
plt.show()


Por último se dibuja, para cada trial, cómo fue cambiando la accuracy de validación época a época, separado por optimizador. Las líneas grises son los trials que cortó el pruning, así se ve en qué época los cortó y que venían por debajo del resto.


## 10. Overfitting


In [ ]:
OVERFIT_CONFIGS = {
    "1 capa x 64": [64],
    "2 capas x 512": [512, 512],
    "3 capas x 1024": [1024, 1024, 1024],
}
EPOCHS_OVERFIT = 15

# Mismo preprocesamiento que el mejor modelo y la misma división train/validación de la búsqueda
train_texts_of, _ = preprocessed_cache[(BEST_REMOVE_SW, BEST_NORM)]
tr_texts, val_texts, tr_labels, val_labels = train_test_split(
    train_texts_of, newsgroups_train.target, test_size=0.2, random_state=SEED,
    stratify=newsgroups_train.target,
)
X_tr, X_val, _ = vectorize_data(tr_texts, val_texts, BEST_NGRAM, BEST_MAX_FEATURES, BEST_MIN_DF)
train_loader_of = DataLoader(to_tensor_dataset(X_tr, tr_labels), batch_size=64, shuffle=True)
val_loader_of = DataLoader(to_tensor_dataset(X_val, val_labels), batch_size=64, shuffle=False)

overfit_results = {}
for name, hidden_sizes in OVERFIT_CONFIGS.items():
    torch.manual_seed(SEED)
    model_of = MLP(X_tr.shape[1], hidden_sizes, NUM_CLASSES, "relu")
    optimizer_of = optim.Adam(model_of.parameters(), lr=1e-3)
    n_params = sum(p.numel() for p in model_of.parameters())
    with mlflow.start_run(run_name=f"overfit_{name}"):
        mlflow.log_params({"hidden_sizes": str(hidden_sizes), "epochs": EPOCHS_OVERFIT})
        start = time.perf_counter()
        hist = train_model(model_of, train_loader_of, val_loader_of, optimizer_of,
                           epochs=EPOCHS_OVERFIT, run_name=name)
        elapsed = time.perf_counter() - start
    overfit_results[name] = {"history": hist, "time": elapsed, "n_params": n_params}


Se entrenan tres redes de distinto tamaño, una chica de 1 capa con 64 neuronas, una mediana de 2 capas con 512 y una grande de 3 capas con 1024, que es el máximo de capas que permite la consigna. Todo lo demás queda fijo para que la única diferencia sea el tamaño: el preprocesamiento del mejor modelo, la misma validación que en la búsqueda, ReLU, Adam con learning rate 0,001 y batch 64. Se entrenan 15 épocas, más que las 5 de la búsqueda, para darle tiempo al overfitting a aparecer. De cada red se guardan las curvas, el tiempo y la cantidad de parámetros.


In [ ]:
fig, axes = plt.subplots(2, len(overfit_results), figsize=(18, 9), sharey="row")
epochs_axis = range(1, EPOCHS_OVERFIT + 1)
for col, (name, r) in enumerate(overfit_results.items()):
    h = r["history"]
    best_ep = int(np.argmin(h["val_loss"])) + 1

    ax = axes[0, col]
    ax.plot(epochs_axis, h["train_loss"], marker="o", ms=3, label="train")
    ax.plot(epochs_axis, h["val_loss"], marker="o", ms=3, label="validación")
    ax.axvline(best_ep, color="gray", linestyle=":", label=f"val loss mínima (época {best_ep})")
    ax.set_title(f"{name}  ({r['n_params']:,} parámetros)")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.7)

    ax = axes[1, col]
    ax.plot(epochs_axis, h["train_acc"], marker="o", ms=3, label="train")
    ax.plot(epochs_axis, h["val_acc"], marker="o", ms=3, label="validación")
    ax.set_xlabel("Época")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.7)

fig.suptitle("Curvas de entrenamiento según el tamaño del MLP", fontsize=14)
plt.tight_layout()
plt.show()

rows = []
for name, r in overfit_results.items():
    h = r["history"]
    best_ep = int(np.argmin(h["val_loss"]))
    rows.append({
        "Modelo": name,
        "Parámetros": r["n_params"],
        "Tiempo (s)": r["time"],
        "Época con val loss mínima": best_ep + 1,
        "Val loss mínima": h["val_loss"][best_ep],
        "Val loss final": h["val_loss"][-1],
        "Acc train final": h["train_acc"][-1],
        "Acc val final": h["val_acc"][-1],
        "Diferencia train - val": h["train_acc"][-1] - h["val_acc"][-1],
    })
display(pd.DataFrame(rows).set_index("Modelo").round(4))


Para cada red se grafica arriba la loss y abajo la accuracy, de train y de validación, época por época. La línea punteada marca la época donde la loss de validación fue la más baja. Si después de ese punto la loss de train sigue bajando pero la de validación sube, la red está memorizando el train en vez de aprender, que es el overfitting. La tabla resume lo mismo en números, con la cantidad de parámetros, el tiempo y la diferencia final entre train y validación.
